# Classification Workflow

This notebook shows how to run an EpiScope classification task and how to define a new classification task with categories that are not built into the package.


## Setup

A classifier needs four pieces: structured paper records, an index/retriever, a classifier config, and a generator that returns JSON matching the config schema.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate `notebooks/episcope_nb.py`, the shared helper module. This works whether
# the kernel starts in `notebooks/` or at the repository root.
_cwd = Path.cwd()
_nb_dir = next(
    (
        directory
        for candidate in [_cwd, *_cwd.parents]
        for directory in (candidate, candidate / "notebooks")
        if (directory / "episcope_nb.py").is_file()
    ),
    None,
)
if _nb_dir is None:
    raise FileNotFoundError("Could not find notebooks/episcope_nb.py")
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

# Importing the helper also puts `src/` on sys.path when this is a checkout.
import episcope_nb as nb

WORK_DIR = nb.bootstrap()
WORK_DIR

## Build A Small Classification Corpus

The examples use two small papers so the retrieved evidence and final labels are easy to inspect.


In [ ]:
sample_papers = nb.classification_papers()
{pid: [s.title for s in p["sections"]] for pid, p in sample_papers.items()}

## Index The Papers

Classification uses the retriever to find evidence for each candidate category. The tiny embedder keeps this notebook offline and deterministic.


In [ ]:
# A vocabulary tuned to the routing categories below, rather than the
# data-sharing one the other notebooks use.
embedder = nb.TinyKeywordEmbedder(
    nb.CLASSIFICATION_VOCABULARY,
    name="tiny-classification-demo",
)
embedder.model_name, embedder.dim

In [ ]:
strategy_name = "classification-demo"
academic_db = nb.build_academic_db(sample_papers, strategy_name)
vdb, retriever = nb.build_index(
    WORK_DIR / "index",
    sample_papers,
    embedder,
    chunk_size=650,
    chunk_overlap=75,
)

len(vdb.get_points()), academic_db.list_docs(strategy_name)

## Run A Built-In Classification Task

Built-in configs live in `episcope.workflows.classification`. This example uses
`DataAccessibilityClassifierConfig`, which classifies how the paper says its data
can be accessed. The generator is the shared `nb.FixedJSONGenerator`, so the
labels below come from a fixed payload rather than a model.

In [ ]:
from episcope.workflows import PaperClassifier
from episcope.workflows.classification import DataAccessibilityClassifierConfig

availability_config = DataAccessibilityClassifierConfig(top_k=4)
availability_payload = {
    "classification": ["A"],
    "primary_label": "A",
    "reasoning": "The paper states that de-identified trial data and code are available in a public repository.",
    "confidence": 0.92,
    "class_probabilities": {
        "A": 0.92,
        "B": 0.03,
        "C": 0.02,
        "D": 0.01,
        "E": 0.01,
        "F": 0.01,
    },
}

availability_classifier = PaperClassifier(
    retriever=retriever,
    generator=nb.FixedJSONGenerator(availability_payload, model_id="demo-data-availability"),
    strategy_name=strategy_name,
    config=availability_config,
    academic_db=academic_db,
)

availability_result = availability_classifier.run_detailed("paper_open_data")
[label.value for label in availability_result.decision.result.classification]

## Inspect The Classification Trace

Use `run_detailed` while developing a task. It exposes retrieved evidence, prompt messages, raw generator output, parsed labels, and training-oriented artifacts.


In [ ]:
print("Classification:", [label.value for label in availability_result.decision.result.classification])
print("Confidence:", availability_result.decision.result.confidence)
print("Reasoning:", availability_result.decision.result.evidence["reasoning"])

print("\nTop evidence")
for chunk in availability_result.decision.top_evidence:
    print(f"- category={chunk.artifacts.get('category')} | {chunk.paper_id} | {chunk.section_title}")
    print(" ", chunk.text[:180])

print("\nRaw generator output")
print(availability_result.trace.raw_llm_response)


## Define A New Task Declaratively (Recommended)

The simplest way to add a classifier is a *declarative task spec*: plain data
describing the label set. EpiScope generates the prompt and output schema for
you, so you do not write any Pydantic models. `multi_label=False` keeps a single
label; set retrieval `examples` per label to improve evidence recall.

In [ ]:
from episcope.workflows.registry import TaskSpec, build_classifier_config_from_spec

review_route_spec = TaskSpec.model_validate(
    {
        "key": "review_route",
        "kind": "classifier",
        "label": "Review route",
        "multi_label": False,
        "default_label": "manual_review",
        "labels": [
            {
                "code": "action_or_policy",
                "name": "Action or policy relevant",
                "definition": "Directly supports action, policy, guidance, or implementation.",
                "examples": ["The findings support a guideline or operational decision."],
            },
            {
                "code": "method_or_tool",
                "name": "Method or tool",
                "definition": "Develops or evaluates a method, tool, model, or workflow.",
                "examples": ["The main contribution is a reusable screening method."],
            },
            {
                "code": "manual_review",
                "name": "Manual review",
                "definition": "Ambiguous or unsupported; needs manual review.",
                "examples": [],
            },
        ],
    }
)

# Build a runnable config from the spec. The prompt and output schema are
# generated automatically from the labels above.
review_route_config = build_classifier_config_from_spec(review_route_spec)
review_route_config.category_labels

In [ ]:
review_payload = {
    "reasoning": "The paper's main contribution is a reusable screening method for prioritizing review records.",
    "classification": ["method_or_tool"],
}

review_classifier = PaperClassifier(
    retriever=retriever,
    generator=nb.FixedJSONGenerator(review_payload, model_id="demo-review-route"),
    strategy_name=strategy_name,
    config=review_route_config,
    academic_db=academic_db,
)

review_result = review_classifier.run_detailed("paper_methods")
review_result.decision.result.classification

## The Same Task From The CLI Or API

A task spec is just data, so you can keep it in a JSON file and use it without
writing Python at all:

- **Scaffold + validate**: `episcope tasks new --kind classifier > review_route.json`,
  then `episcope tasks validate --task-file review_route.json`. Add `--interactive`
  to `tasks new` to answer prompts instead of editing the scaffold by hand.
- **Run from the CLI**: `episcope classify --file paper.pdf --task-file review_route.json`,
  or drop the file in `<workspace>/tasks/` and select it by key with `--classifier-kind`.
- **Run via the API**: send the spec inline as the `task` field of a `/classify` request.

See the [Declarative Tasks](https://github.com/VinsRR/EpiScope/wiki/Declarative-Tasks)
wiki page for the full spec format and all delivery options. The miner equivalent
is in `06_data_source_extraction.ipynb`.

## Advanced: Build A Config By Hand (Under The Hood)

The declarative spec above builds a `BaseClassifierConfig` for you. You can also
assemble one directly when you need custom Pydantic validation, a bespoke output
schema, or full control over the prompt. This is the low-level API the
declarative path is built on.

In [ ]:
from typing import List, Literal, Optional

from pydantic import Field, model_validator

from episcope.workflows.classification import BaseClassifierConfig
from episcope.workflows.classification.schemas import BaseClassificationSchema

ReviewRouteCode = Literal["A", "B", "C"]


class ReviewRouteOutput(BaseClassificationSchema):
    classification: List[ReviewRouteCode] = Field(
        ...,
        min_length=1,
        description="A list with one review-route code: A, B, or C.",
    )
    primary_label: Optional[ReviewRouteCode] = Field(
        default=None,
        description="The dominant review-route code.",
    )

    @model_validator(mode="after")
    def normalize(self) -> "ReviewRouteOutput":
        canonical = [code for code in ["A", "B", "C"] if code in self.classification]
        if not canonical:
            canonical = ["C"]
        self.classification = [canonical[0]]
        self.primary_label = self.primary_label if self.primary_label in self.classification else self.classification[0]
        return self


ReviewRouteOutput.model_rebuild()


review_route_labels = {
    "A": "Action or policy relevant",
    "B": "Method or tool",
    "C": "Manual review",
}
review_route_definitions = {
    "A": "Directly supports action, policy, guidance, or implementation.",
    "B": "Develops or evaluates a method, tool, model, or workflow.",
    "C": "Needs manual review because the route is ambiguous or unsupported.",
}

custom_config = BaseClassifierConfig(
    top_k=4,
    template_paragraphs={
        "action_or_policy": [
            "The findings support a policy, guideline, implementation plan, or operational decision.",
            "The paper provides evidence that can be used for public health action or clinical workflow changes.",
        ],
        "method_or_tool": [
            "The main contribution is a method, tool, model, screening system, or reusable workflow.",
            "The paper develops or evaluates a methodological approach rather than making a direct policy recommendation.",
        ],
        "manual_review": [
            "The paper is ambiguous, background-only, or lacks enough information for automatic routing.",
        ],
    },
    classification_mapping={
        "A": "action_or_policy",
        "B": "method_or_tool",
        "C": "manual_review",
    },
    category_labels=review_route_labels,
    system_prompt=(
        "You route epidemiology papers for review. Choose the route that best reflects "
        "the paper's main contribution, using only the provided text."
    ),
    user_prompt_template=r'''
Classify the paper into exactly one review route.

Categories:
{categories}

Definitions:
{definitions}

Paper content:
Title: {title}
Abstract: {abstract}
Keywords: {keywords}

Relevant extracts:
{chunks_info}

Instructions:
1. Choose A when the paper directly supports action, policy, guidance, or implementation.
2. Choose B when the main contribution is a method, tool, model, or reusable workflow.
3. Choose C when the evidence is insufficient or the paper needs manual review.
4. Return only a JSON object matching this schema:
{schema}
''',
    extra_output_fields={
        "definitions": "\n".join(
            f"{code}: {definition}" for code, definition in review_route_definitions.items()
        )
    },
    output_schema=ReviewRouteOutput,
    default_classification=["manual_review"],
)

custom_config.category_labels

## Run The Custom Classifier

The workflow is the same as the built-in task. Only the config, schema, and expected generator JSON changed.


In [ ]:
custom_payload = {
    "classification": ["B"],
    "primary_label": "B",
    "reasoning": "The paper's main contribution is a reusable screening method for prioritizing review records.",
    "confidence": 0.88,
    "class_probabilities": {"A": 0.08, "B": 0.88, "C": 0.04},
}

custom_classifier = PaperClassifier(
    retriever=retriever,
    generator=nb.FixedJSONGenerator(custom_payload, model_id="demo-review-route"),
    strategy_name=strategy_name,
    config=custom_config,
    academic_db=academic_db,
)

custom_result = custom_classifier.run_detailed("paper_methods")
custom_result.decision.result.classification

In [ ]:
print("Classification:", custom_result.decision.result.classification)
print("Reasoning:", custom_result.decision.result.evidence["reasoning"])
print("Prompt messages:", len(custom_result.trace.prompt_messages))
print("Raw JSON:", custom_result.trace.raw_llm_response)

print("\nEvidence categories retrieved for the task")
for chunk in custom_result.decision.top_evidence:
    print(f"- {chunk.artifacts.get('category')} | {chunk.section_title}: {chunk.text[:150]}")


For real model calls, replace `nb.FixedJSONGenerator` with `LLMGenerator`